# 03 · Evaluation on 2024 (out-of-time test)

1. Predictions on the test year
2. Ranking: AUC, Gini, lift
3. Calibration
4. Explaining LightGBM with SHAP
5. Fairness check: with vs. without gender
6. Risk tiers
7. Final model choice

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # make `src` importable from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config

from sklearn.metrics import roc_auc_score
from src.train import fit_glm, fit_lgbm
from src.evaluate import gini, lift_table, calibration_table, assign_risk_tier

In [ ]:
train = pd.read_csv(config.TRAIN_FILE)
test = pd.read_csv(config.TEST_FILE)

glm = fit_glm(train)
lgbm = fit_lgbm(train)

y = test[config.TARGET]
p_glm = glm.predict(test, offset=np.log(test[config.EXPOSURE_COL]))
p_lgbm = lgbm.predict_proba(test[config.FEATURES + [config.EXPOSURE_COL]])[:, 1]

## 2. Ranking metrics

In [ ]:
pd.DataFrame({
    "AUC": [roc_auc_score(y, p_glm), roc_auc_score(y, p_lgbm)],
    "Gini": [gini(y, p_glm), gini(y, p_lgbm)],
}, index=["GLM", "LightGBM"]).round(3)

In [ ]:
lift_table(y, p_glm, test[config.EXPOSURE_COL])

## 3. Calibration

The app shows probabilities to users, so they need to be correct, not just well ranked.

In [ ]:
cal = calibration_table(y, p_glm)
plt.plot(cal["predicted_rate"], cal["actual_rate"], "o-", label="GLM")
lim = [0, cal[["predicted_rate", "actual_rate"]].max().max() * 1.1]
plt.plot(lim, lim, "--", color="grey", label="Perfect calibration")
plt.xlabel("Mean predicted probability"); plt.ylabel("Observed claim rate"); plt.legend()
# TODO: add LightGBM, save to config.FIGURES_DIR

## 4. SHAP for LightGBM

In [ ]:
# import shap
# X_test = lgbm.named_steps["preprocess"].transform(test[config.FEATURES + [config.EXPOSURE_COL]])
# explainer = shap.TreeExplainer(lgbm.named_steps["model"])
# shap_values = explainer.shap_values(X_test)
# shap.summary_plot(shap_values, X_test, feature_names=lgbm.named_steps["preprocess"].get_feature_names_out())

## 5. Fairness check

Since 2012, EU insurers cannot use gender to set prices. TODO: refit with and without gender (if available). How much does performance change? Does removing it shift predictions for other groups (proxy effects)?

## 6. Risk tiers

TODO: choose cutoffs, then update `RISK_TIER_CUTOFFS` in `src/config.py`.

In [ ]:
tiers = assign_risk_tier(p_glm, config.RISK_TIER_CUTOFFS)
pd.DataFrame({"tier": tiers, "actual": y}).groupby("tier").agg(
    policies=("actual", "size"), actual_claim_rate=("actual", "mean"))

## 7. Final model choice

TODO: which model goes in the app, and why? (accuracy vs. interpretability vs. calibration)